# B2-022-probabilistic-latent-models — Practice p17 — Solution

**Type:** integrative · **Difficulty:** core · **Concepts:** autoencoder

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20260927
LOADER_CANDIDATES = (
    Path("../data/latent_data.py"),
    Path("units/B2-022-probabilistic-latent-models/data/latent_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("latent_data", loader_path)
latent_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(latent_data)
torch.set_num_threads(1)

## Solution

Plan: build the pinned split readers, construct the model right after `torch.manual_seed(SEED)`, measure held-out MSE before training, run exactly 400 full-batch Adam updates on the training rows only, then measure held-out MSE again.
A forward pre-hook is installed **only for the training call** and hashes every row the model's `forward` receives, proving that only training rows enter training; it is removed before the held-out evaluation.

In [ ]:
import copy

DATASET = "mixture8d"


def train_rows():
    return latent_data.train_tensor(DATASET)


def heldout_rows():
    return latent_data.heldout_tensor(DATASET)


def split_hash_sets(name):
    hashes = latent_data.ROW_SHA256[name]
    train_part = {hashes[i] for i in latent_data.TRAIN_IDS[name]}
    heldout_part = {hashes[i] for i in latent_data.HELDOUT_IDS[name]}
    return train_part, heldout_part


TRAIN_HASHES, HELDOUT_HASHES = split_hash_sets(DATASET)


def install_training_row_guard(model):
    """Forward pre-hook: every row a training forward receives must hash into the training part."""
    seen = {"calls": 0, "rows": 0}

    def hook(module, args):
        batch = args[0]
        for row in batch:
            row_hash = latent_data.row_sha256(row)
            if row_hash not in TRAIN_HASHES or row_hash in HELDOUT_HASHES:
                raise AssertionError("a non-training row reached a training forward pass")
        seen["calls"] += 1
        seen["rows"] += batch.shape[0]

    return model.register_forward_pre_hook(hook), seen


class TinyAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 2))
        self.decoder = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 8))

    def encode(self, x):
        return self.encoder(x)

    def decode(self, h):
        return self.decoder(h)

    def forward(self, x):
        return self.decode(self.encode(x))


def train_autoencoder(model, batch, optimizer):
    trace = []
    for _ in range(400):
        optimizer.zero_grad(set_to_none=True)
        reconstruction = model(batch)
        loss = F.mse_loss(reconstruction, batch)
        trace.append(loss.item())
        loss.backward()
        optimizer.step()
    return trace


torch.manual_seed(SEED)
model = TinyAutoencoder()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)

# (A)
H = heldout_rows()
with torch.no_grad():
    heldout_before = F.mse_loss(model(H), H).item()
params_before = copy.deepcopy({name: p.detach() for name, p in model.named_parameters()})

# (B)
guard, guard_stats = install_training_row_guard(model)
trace = train_autoencoder(model, train_rows(), optimizer)
guard.remove()
with torch.no_grad():
    heldout_after = F.mse_loss(model(H), H).item()
    heldout_codes = model.encode(H)
    baseline_mse = F.mse_loss(train_rows().mean(dim=0).expand_as(H), H).item()

changed = [name for name, p in model.named_parameters() if not torch.equal(p.detach(), params_before[name])]
print(f"trace[0]={trace[0]:.4f} trace[-1]={trace[-1]:.5f}")
print(f"heldout before={heldout_before:.4f} after={heldout_after:.5f} ratio={heldout_after / heldout_before:.5f}")
print(f"train-mean baseline held-out MSE={baseline_mse:.4f}; guarded rows={guard_stats}")

### (D) Why the held-out comparison is the certificate

The training trace only shows that the optimizer fit the 48 rows it was given, which a large enough model can do by memorization, whereas the held-out MSE is measured on 12 rows that never touched a forward pass in training, a gradient, or a choice, so its drop (here from about 1.71 to about 0.0024) is evidence that the learned 2-D code generalizes.
The natural trivial baseline, predicting the training mean for every held-out row, has held-out MSE about 1.24, so an autoencoder must route real per-row information through its 2-number bottleneck to get anywhere near the bound.
The absolute bound `0.05` is therefore about 4% of the baseline error, which rules out a collapsed or mean-predicting model even though the relative `10%` bound alone could in principle be met by a poor initialization that improved a lot.

### Answer check

In [ ]:
T = train_rows()
assert T.shape == (48, 8) and H.shape == (12, 8)
assert T.dtype == torch.float32 and H.dtype == torch.float32
assert tuple(heldout_codes.shape) == (12, 2)
assert len(trace) == 400 and all(isinstance(v, float) for v in trace)
assert trace[-1] < trace[0]
assert heldout_after <= 0.10 * heldout_before
assert heldout_after <= 0.05
assert len(changed) >= 1
assert guard_stats == {"calls": 400, "rows": 400 * 48}
assert all(latent_data.row_sha256(row) in TRAIN_HASHES for row in T)
assert not any(latent_data.row_sha256(row) in HELDOUT_HASHES for row in T)
assert baseline_mse > 0.05